# 04 — Correlação, robustez e discussão clínica (T07 e T15)

Complementa a EDA (`AnaliseExploratoria.ipynb`) e os notebooks 02 e 03. Split e modelos reproduzidos com os mesmos parâmetros (`random_state=42`). Classe positiva: **M (maligno)**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

df = pd.read_csv("../data/data.csv").drop(columns=['id', 'Unnamed: 32'])
X = df.drop(columns=['diagnosis'])
y = df['diagnosis']

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, stratify=y, random_state=42)

def make_model(classifier):
    pre = ColumnTransformer(transformers=[('num', Pipeline(steps=[('scaler', StandardScaler())]), list(X.columns))])
    return Pipeline(steps=[('preprocessor', pre), ('classifier', classifier)])

modelos = {
    'Regressão Logística': make_model(LogisticRegression(random_state=42)),
    'Random Forest': make_model(RandomForestClassifier(random_state=42)),
    'KNN': make_model(KNeighborsClassifier()),
}

## T07 — Análise de correlação

In [ ]:
corr = X.corr()
pares = (corr.where(np.triu(np.ones(corr.shape, dtype=bool), k=1))
             .stack().dropna().rename('correlação').reset_index()
             .rename(columns={'level_0': 'feature A', 'level_1': 'feature B'}))
pares['|r|'] = pares['correlação'].abs()
top_pares = pares.sort_values('|r|', ascending=False).head(15).drop(columns='|r|')
print(f"Pares com |r| > 0,9: {(pares['|r|'] > 0.9).sum()} de {len(pares)}")
top_pares.round(3).reset_index(drop=True)

In [ ]:
y_bin = (y == 'M').astype(int)
corr_alvo = X.corrwith(y_bin).sort_values(key=np.abs, ascending=False)
corr_alvo.head(10).round(3)

In [ ]:
plt.figure(figsize=(7, 4))
corr_alvo.sort_values().tail(10).plot.barh(title="Correlação com o alvo (M=1) — top 10")
plt.tight_layout()
plt.savefig("../reports/figures/correlacao_alvo.png", dpi=150)
plt.show()

## Robustez — validação cruzada estratificada\n\nUsa apenas treino + validação (85% dos dados); o conjunto de teste não é tocado. Mostra a média e o desvio das métricas em 10 folds, para avaliar a estabilidade dos resultados do T12.

In [ ]:
cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)
scoring = {'accuracy': 'accuracy', 'recall_M': 'recall', 'f1_M': 'f1'}
y_temp_bin = (y_temp == 'M').astype(int)   # M=1 para o scorer usar pos_label=1

linhas = []
for nome, m in modelos.items():
    r = cross_validate(m, X_temp, y_temp_bin, cv=cv, scoring=scoring)
    linhas.append({
        'Modelo': nome,
        'Accuracy': f"{r['test_accuracy'].mean():.3f} ± {r['test_accuracy'].std():.3f}",
        'Recall (M)': f"{r['test_recall_M'].mean():.3f} ± {r['test_recall_M'].std():.3f}",
        'F1 (M)': f"{r['test_f1_M'].mean():.3f} ± {r['test_f1_M'].std():.3f}",
    })
pd.DataFrame(linhas).set_index('Modelo')

## Conclusões T07 e validação cruzada

**Correlação**
- 21 dos 435 pares de features têm |r| > 0,9. Os mais altos são `radius`, `perimeter` e `area`, que medem o tamanho do núcleo: `radius_mean` × `perimeter_mean` = 0,998, e há a mesma redundância entre as versões `mean` e `worst` (`perimeter_mean` × `perimeter_worst` = 0,970).
- As features mais associadas ao diagnóstico são `concave points_worst` (0,79), `perimeter_worst` (0,78), `concave points_mean` (0,78) e `radius_worst` (0,78), as mesmas apontadas por SHAP e importância no notebook 03. Ou seja, os dois métodos independentes concordam.
- **Consequência prática:** a informação de "tamanho" está repetida em até 9 colunas. Isso não prejudica a acurácia dos modelos, mas explica os coeficientes instáveis da Regressão Logística e a importância dispersa por permutação. Reduzir dimensionalidade (PCA) ou remover features redundantes é uma melhoria possível, não aplicada aqui para preservar o pipeline do T06.

**Validação cruzada (10 folds, treino + validação)**

| Modelo | Accuracy | Recall (M) | F1 (M) |
|---|---|---|---|
| Regressão Logística | 0,973 ± 0,023 | 0,956 ± 0,042 | 0,963 ± 0,031 |
| Random Forest | 0,956 ± 0,024 | 0,928 ± 0,043 | 0,941 ± 0,031 |
| KNN | 0,965 ± 0,021 | 0,911 ± 0,044 | 0,950 ± 0,030 |

- A Regressão Logística fica à frente nas três métricas, como no teste do T12, o que dá mais confiança de que a ordenação não foi acaso do split.
- Ainda assim, o desvio de ~0,04 no recall é do tamanho da diferença entre modelos (0,956 vs 0,928): a vantagem sobre o Random Forest é sugestiva, não conclusiva. O KNN tem o menor recall médio (0,911), o pior para triagem, apesar de acurácia parecida.


## T15 — Discussão crítica de uso prático

**Papel do modelo.** O modelo é uma ferramenta de apoio à triagem: sinaliza casos que merecem atenção prioritária, mas não emite diagnóstico. O médico integra o resultado com histórico, exames de imagem e biópsia, e tem a palavra final.

**Qual erro importa mais.** No teste, a Regressão Logística errou 1 dos 32 casos malignos (recall 0,969) e não gerou falsos positivos; Random Forest e KNN deixaram passar 3 malignos cada (recall 0,906). Um falso negativo atrasa o diagnóstico de um câncer; um falso positivo gera um exame adicional e ansiedade. Por isso o recall da classe maligna é a métrica de referência, e o modelo com maior recall (Regressão Logística) é o candidato mais adequado. Se o hospital quiser reduzir ainda mais os falsos negativos, o limiar de decisão pode ser rebaixado (por exemplo, sinalizar como suspeito com probabilidade de maligno menor que 0,5), aceitando mais falsos positivos. Esse ajuste não foi feito neste trabalho e exigiria definir com a equipe clínica quanto de falso positivo é aceitável.

**Interpretabilidade.** SHAP mostra que o modelo se apoia em tamanho e concavidade do núcleo (`area_worst`, `concave points_worst`, `perimeter_worst`), coerente com o conhecimento clínico e com a correlação dessas features com o diagnóstico. Isso permite ao médico conferir se a justificativa faz sentido. Porém é a explicação do modelo, não da doença: são associações aprendidas nesta base, sem garantia de causalidade.

**Limitações.**
- Base pequena (569 casos) e de uma única origem (Breast Cancer Wisconsin, features extraídas de imagens digitalizadas de punção aspirativa). Nada garante o mesmo desempenho em outra população, equipamento ou protocolo de coleta.
- Poucos casos malignos no teste (32): as métricas têm intervalos largos, como mostra a validação cruzada (±0,04 no recall).
- Features altamente redundantes (T07); o desempenho vem de um sinal forte e simples, o que pode não se manter em casos ambíguos ou raros.
- Não foi feita calibração de probabilidades nem análise por subgrupos.

**Antes de qualquer uso real:** validação externa em dados de outro serviço, estudo prospectivo comparando a triagem com e sem o modelo, aprovação regulatória aplicável a software de apoio à decisão clínica, monitoramento de desempenho ao longo do tempo e tratamento adequado dos dados de pacientes (LGPD). Este trabalho usa somente dado público e é um exercício acadêmico.
